# Importing dependencies and data

In [1]:
import logging
from logging.handlers import RotatingFileHandler
import sys
import pandas as pd
from datetime import datetime

In [2]:


def setup_logging(log_file='pipeline.log'):
    """Configure logging for the pipeline"""
    logger = logging.getLogger('market_analysis')
    logger.setLevel(logging.DEBUG)
    
    # Clear existing handlers
    logger.handlers = []
    
    # Format for our log messages
    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Console handler (shows INFO and above)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)
    
    # File handler (logs everything DEBUG and above)
    file_handler = RotatingFileHandler(
        log_file,
        maxBytes=10*1024*1024,  # 10MB
        backupCount=5
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)
    
    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    
    return logger

logger = setup_logging()

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from lead_lag import lag # For Hayashi-Yoshida estimator [2]
from sklearn.svm import SVC # For SVM model [3, 4]
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.signal import medfilt # For median filter [5]

PATH_FUTURES = '../data/trb_usdt_futures_export.csv'
PATH_SPOT = '../data/trb_usdt_spot_export.csv'
PATH_TRADES = '../data/trb_usdt_trades_export.csv'

def load_data(path):
    df = pd.read_csv(path, parse_dates=['time'])
    df.rename(columns={'time': 'timestamp', 'bid_price':'best_bid', 'ask_price':'best_ask'}, inplace=True)
    try:
        # Correct: Pass the 'timestamp' Series to to_datetime
        df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601', utc=True)
    except ValueError:
        # Fallback for mixed formats
        try:
            df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True)
        except Exception as e:
            # logger.error(f"Failed to parse timestamps in {path}: {str(e)}")
            # Re-raising the exception is good practice if you can't handle it
            raise

    df = df.set_index('timestamp').sort_index()
    return df


futures = load_data(PATH_FUTURES)
spot = load_data(PATH_SPOT)
trades = load_data(PATH_TRADES)

In [7]:
spot

,symbol,best_bid,best_ask
timestamp,,,
2025-05-27 12:00:00.354000+00:00,TRB_USDT,43.15,43.17
2025-05-27 12:00:00.360000+00:00,TRB_USDT,43.16,43.17
2025-05-27 12:00:00.382000+00:00,TRB_USDT,43.15,43.17
2025-05-27 12:00:00.384000+00:00,TRB_USDT,43.16,43.17
2025-05-27 12:00:00.391000+00:00,TRB_USDT,43.15,43.17
...,...,...,...
2025-05-28 11:48:50.829000+00:00,TRB_USDT,50.56,50.57
2025-05-28 11:48:51.193000+00:00,TRB_USDT,50.56,50.58
2025-05-28 11:48:51.198000+00:00,TRB_USDT,50.57,50.58


In [8]:
futures

,symbol,best_bid,best_ask
timestamp,,,
2025-05-27 12:00:00.025000+00:00,TRB_USDT,43.140,43.141
2025-05-27 12:00:00.029000+00:00,TRB_USDT,43.138,43.140
2025-05-27 12:00:00.029000+00:00,TRB_USDT,43.138,43.141
2025-05-27 12:00:02.470000+00:00,TRB_USDT,43.134,43.140
2025-05-27 12:00:02.478000+00:00,TRB_USDT,43.133,43.137
...,...,...,...
2025-05-28 11:48:50.551000+00:00,TRB_USDT,50.537,50.544
2025-05-28 11:48:50.554000+00:00,TRB_USDT,50.537,50.542
2025-05-28 11:48:50.562000+00:00,TRB_USDT,50.537,50.541


In [9]:
trades

,symbol,price,quantity,is_market_maker
timestamp,,,,
2025-05-27 12:00:00.382000+00:00,TRB_USDT,43.16,6.750,t
2025-05-27 12:00:00.523000+00:00,TRB_USDT,43.15,7.950,t
2025-05-27 12:00:01.824000+00:00,TRB_USDT,43.17,0.199,t
2025-05-27 12:00:02.490000+00:00,TRB_USDT,43.16,1.010,t
2025-05-27 12:00:04.225000+00:00,TRB_USDT,43.15,4.678,t
...,...,...,...,...
2025-05-28 11:48:44.298000+00:00,TRB_USDT,50.57,6.750,t
2025-05-28 11:48:44.298000+00:00,TRB_USDT,50.56,6.750,t
2025-05-28 11:48:45.095000+00:00,TRB_USDT,50.54,5.600,t


# --- 1. Data Pre-processing and Feature Engineering ---


#### 1.1 Addressing Irregular Timestamps and Data Synchronization Challenges

#### 1.2 Definition and Calculation of Mid-Price, Bid/Ask Spreads, and Other Microstructure Features


In [10]:
spot_data = spot.copy()
perpetual_data = futures.copy()

spot_data['bid_ask_spread'] = spot_data['best_ask'] - spot_data['best_bid']
spot_data['mid_price'] = (spot_data['best_ask'] + spot_data['best_bid']) / 2
perpetual_data['bid_ask_spread'] = perpetual_data['best_ask'] - perpetual_data['best_bid']
perpetual_data['mid_price'] = (perpetual_data['best_ask'] + perpetual_data['best_bid']) / 2

# Example of a "time-sensitive" feature: Price change over a short window (e.g., 1ms)
# This requires careful handling of irregular timestamps. For simplicity, we'll use `diff`
# and then consider the time difference.
spot_data['mid_price_change_1ms'] = spot_data['mid_price'].diff()
perpetual_data['mid_price_change_1ms'] = perpetual_data['mid_price'].diff()

# The user query mentions: "when prices are moving upward, the ask side tends to update first;
# conversely, when prices are declining, the bid side often reacts earlier."
# This can be incorporated by creating features that capture these asymmetric movements.
# For example, tracking changes in best_ask vs best_bid separately.
spot_data['best_bid_change'] = spot_data['best_bid'].diff()
spot_data['best_ask_change'] = spot_data['best_ask'].diff()
perpetual_data['best_bid_change'] = perpetual_data['best_bid'].diff()
perpetual_data['best_ask_change'] = perpetual_data['best_ask'].diff()

In [11]:
spot_data

,symbol,best_bid,best_ask,bid_ask_spread,mid_price,mid_price_change_1ms,best_bid_change,best_ask_change
timestamp,,,,,,,,
2025-05-27 12:00:00.354000+00:00,TRB_USDT,43.15,43.17,0.02,43.160,NaN,NaN,NaN
2025-05-27 12:00:00.360000+00:00,TRB_USDT,43.16,43.17,0.01,43.165,0.005,0.01,0.00
2025-05-27 12:00:00.382000+00:00,TRB_USDT,43.15,43.17,0.02,43.160,-0.005,-0.01,0.00
2025-05-27 12:00:00.384000+00:00,TRB_USDT,43.16,43.17,0.01,43.165,0.005,0.01,0.00
2025-05-27 12:00:00.391000+00:00,TRB_USDT,43.15,43.17,0.02,43.160,-0.005,-0.01,0.00
...,...,...,...,...,...,...,...,...
2025-05-28 11:48:50.829000+00:00,TRB_USDT,50.56,50.57,0.01,50.565,0.005,0.01,0.00
2025-05-28 11:48:51.193000+00:00,TRB_USDT,50.56,50.58,0.02,50.570,0.005,0.00,0.01
2025-05-28 11:48:51.198000+00:00,TRB_USDT,50.57,50.58,0.01,50.575,0.005,0.01,0.00


In [12]:
perpetual_data

,symbol,best_bid,best_ask,bid_ask_spread,mid_price,mid_price_change_1ms,best_bid_change,best_ask_change
timestamp,,,,,,,,
2025-05-27 12:00:00.025000+00:00,TRB_USDT,43.140,43.141,0.001,43.1405,NaN,NaN,NaN
2025-05-27 12:00:00.029000+00:00,TRB_USDT,43.138,43.140,0.002,43.1390,-0.0015,-0.002,-0.001
2025-05-27 12:00:00.029000+00:00,TRB_USDT,43.138,43.141,0.003,43.1395,0.0005,0.000,0.001
2025-05-27 12:00:02.470000+00:00,TRB_USDT,43.134,43.140,0.006,43.1370,-0.0025,-0.004,-0.001
2025-05-27 12:00:02.478000+00:00,TRB_USDT,43.133,43.137,0.004,43.1350,-0.0020,-0.001,-0.003
...,...,...,...,...,...,...,...,...
2025-05-28 11:48:50.551000+00:00,TRB_USDT,50.537,50.544,0.007,50.5405,0.0010,0.000,0.002
2025-05-28 11:48:50.554000+00:00,TRB_USDT,50.537,50.542,0.005,50.5395,-0.0010,0.000,-0.002
2025-05-28 11:48:50.562000+00:00,TRB_USDT,50.537,50.541,0.004,50.5390,-0.0005,0.000,-0.001


#### 1.3 Leveraging Trade Data: Calculation and Predictive Power of Order Flow Imbalance (OFI)




In [13]:
def classify_trade_direction(trades_df, quotes_df):
    """
    Classifies trades as buyer-initiated (1) or seller-initiated (-1) using a simplified quote test.
    Trades at or above ask are buys, at or below bid are sells.[16, 15]
    Requires aligning trades with the nearest quotes.
    """
    # Merge trades with quotes using merge_asof to get the quote *just before* each trade
    # This is crucial for accurate classification.[16]
    merged_df = pd.merge_asof(trades_df.sort_index(), quotes_df[['best_bid', 'best_ask']].sort_index(),
                              left_index=True, right_index=True, direction='backward')
    
    trade_direction = pd.Series(0, index=merged_df.index)
    
    # Buyer-initiated if trade price >= best_ask
    trade_direction[merged_df['price'] >= merged_df['best_ask']] = 1
    # Seller-initiated if trade price <= best_bid
    trade_direction[merged_df['price'] <= merged_df['best_bid']] = -1
    
    # Trades between bid and ask (or at mid-point) are harder to classify without more info.[16]
    # For simplicity, we leave them as 0 or could use a tick test (previous trade price) if needed.
    
    return trade_direction

def calculate_trade_based_ofi(trades_df, quotes_df):
    """
    Calculates Order Flow Imbalance (OFI) based on classified trade direction and quantity.
    This is a simplified version, often OFI uses order book depth changes.[17]
    """
    df_copy = trades_df.copy()
    
    # Classify trade direction for spot data
    df_copy['trade_direction'] = classify_trade_direction(df_copy, quotes_df) # Using spot's own quotes
    
    # Calculate signed trade volume: quantity * direction
    df_copy['signed_trade_volume'] = df_copy['quantity'] * df_copy['trade_direction']
    
    # OFI can be a cumulative sum or a rolling sum of signed trade volume over a short window.[17]
    # A rolling sum (e.g., over 100ms) can capture recent imbalance.
    # For irregular data, a time-based rolling window is better than a fixed number of periods.
    df_copy['OFI'] = df_copy['signed_trade_volume'].rolling('100ms', closed='left').sum().fillna(0) # [17]
    
    return df_copy

ofi_data = calculate_trade_based_ofi(trades, spot_data)

# 2. To analyze mid_price (from quotes) and OFI (from trades) together,
#    we need to merge them. An as-of merge is ideal for this.
#    This finds the most recent OFI for each quote's timestamp.
#    (This assumes both DataFrames have a sorted DatetimeIndex).
combined_data = pd.merge_asof(
    left=spot_data,
    right=ofi_data[['OFI']],  # We only need the OFI column from the result
    left_index=True,
    right_index=True,
    direction='backward'  # Use the last known OFI before the quote
)

print("\nSpot Data with OFI (head):")
print(combined_data[['mid_price', 'OFI']].head())


Spot Data with OFI (head):
                                  mid_price  OFI
timestamp                                       
2025-05-27 12:00:00.354000+00:00     43.160  NaN
2025-05-27 12:00:00.360000+00:00     43.165  NaN
2025-05-27 12:00:00.382000+00:00     43.160  0.0
2025-05-27 12:00:00.384000+00:00     43.165  0.0
2025-05-27 12:00:00.391000+00:00     43.160  0.0


# --- 2. Quantifying Lead-Lag Relationships and Price Discovery ---


#### 2.1 Methodology for Detecting Lead-Lag Relationships Between Spot and Perpetual Markets

In [14]:
spot_series = combined_data['mid_price']
perpetual_series = perpetual_data['mid_price']

spot_series.index = pd.to_datetime(spot_series.index)
perpetual_series.index = pd.to_datetime(perpetual_series.index)

from lead_lag import LeadLag
import numpy as np



# 1. Manually instantiate the LeadLag object.
#    This runs the Python part of the setup, which requires a DatetimeIndex
#    but incorrectly creates 64-bit integer arrays for the timestamps.
ll = LeadLag(spot_series, perpetual_series, max_lag=0.01)

# 2. Manually downcast the internal timestamp arrays to 32-bit integers.
#    This is what the compiled Cython code expects on Windows.
#    This is safe as long as your timestamps are between ~1902 and ~2038.
ll.cc.t_x = np.array(ll.cc.t_x, dtype=np.int32)
ll.cc.t_y = np.array(ll.cc.t_y, dtype=np.int32)


# 3. Run the inference step, which will now receive the correct dtype.
ll.run_inference()
estimated_lag_spot_perpetual = ll.lead_lag

if estimated_lag_spot_perpetual is not None:
    if estimated_lag_spot_perpetual > 0:
        print("Interpretation: Spot market leads the Perpetual market (positive lag means ts1 leads ts2). [4]")
    elif estimated_lag_spot_perpetual < 0:
        print("Interpretation: Perpetual market leads the Spot market (negative lag means ts2 leads ts1). [4]")
    else:
        print("Interpretation: No significant lead-lag detected within the max_lag window.")
else:
    print("Could not estimate lead-lag. Ensure 'lead_lag' library is installed and data is sufficient.")


Interpretation: Spot market leads the Perpetual market (positive lag means ts1 leads ts2). [4]


#### 2.2 Analysis of Which Market Drives Price Discovery More Often


In [15]:
combined_data

,symbol,best_bid,best_ask,bid_ask_spread,mid_price,mid_price_change_1ms,best_bid_change,best_ask_change,OFI
timestamp,,,,,,,,,
2025-05-27 12:00:00.354000+00:00,TRB_USDT,43.15,43.17,0.02,43.160,NaN,NaN,NaN,NaN
2025-05-27 12:00:00.360000+00:00,TRB_USDT,43.16,43.17,0.01,43.165,0.005,0.01,0.00,NaN
2025-05-27 12:00:00.382000+00:00,TRB_USDT,43.15,43.17,0.02,43.160,-0.005,-0.01,0.00,0.0
2025-05-27 12:00:00.384000+00:00,TRB_USDT,43.16,43.17,0.01,43.165,0.005,0.01,0.00,0.0
2025-05-27 12:00:00.391000+00:00,TRB_USDT,43.15,43.17,0.02,43.160,-0.005,-0.01,0.00,0.0
...,...,...,...,...,...,...,...,...,...
2025-05-28 11:48:50.829000+00:00,TRB_USDT,50.56,50.57,0.01,50.565,0.005,0.01,0.00,0.0
2025-05-28 11:48:51.193000+00:00,TRB_USDT,50.56,50.58,0.02,50.570,0.005,0.00,0.01,0.0
2025-05-28 11:48:51.198000+00:00,TRB_USDT,50.57,50.58,0.01,50.575,0.005,0.01,0.00,0.0


In [17]:
# To fill the table from the research, you would need to:
# 1. Run the `lag` function for different price components (mid-price, best bid, best ask).
# 2. Iterate this analysis over different time windows (e.g., hourly, or during high/low volatility periods)
#    of your 24-hour dataset to capture intraday seasonality.[2, 18]
# 3. Calculate liquidity metrics (average bid-ask spread, average trade volume) for each window.

# Example: Calculate average liquidity metrics for the entire dataset
avg_spot_spread = spot_data['bid_ask_spread'].mean()
avg_perpetual_spread = perpetual_data['bid_ask_spread'].mean()
# avg_spot_trade_volume = spot_data[''].mean() # Only spot has trade quantity

print(f"\nAverage Spot Bid-Ask Spread: {avg_spot_spread:.4f}")
print(f"Average Perpetual Bid-Ask Spread: {avg_perpetual_spread:.4f}")
# print(f"Average Spot Trade Volume: {avg_spot_trade_volume:.2f}")


Average Spot Bid-Ask Spread: 0.0217
Average Perpetual Bid-Ask Spread: 0.0094


# --- 3. Predictive Modeling for Cross-Market Trading Signals ---


#### 3.1 Defining and Detecting "Sudden Price Changes"


In [18]:
def detect_sudden_price_change(df, price_col, threshold_bps=7, time_window_ms=3):
    df_copy = df.copy()
    df_copy['price_prev'] = df_copy[price_col].shift(1)
    df_copy['time_prev'] = df_copy.index.to_series().shift(1)
    
    df_copy['pct_change'] = (df_copy[price_col] - df_copy['price_prev']) / df_copy['price_prev'] * 10000
    df_copy['time_diff_ms'] = (df_copy.index.to_series() - df_copy['time_prev']).dt.total_seconds() * 1000
    
    sudden_change_mask = (df_copy['time_diff_ms'] <= time_window_ms) & \
                         (df_copy['pct_change'].abs() >= threshold_bps)
    
    return sudden_change_mask.fillna(False)

spot_data['sudden_change'] = detect_sudden_price_change(spot_data, 'mid_price')
perpetual_data['sudden_change'] = detect_sudden_price_change(perpetual_data, 'mid_price')

logger.info(f"\nNumber of sudden price changes in Spot data: {spot_data['sudden_change'].sum()}")
logger.info(f"Number of sudden price changes in Perpetual data: {perpetual_data['sudden_change'].sum()}")


2026-01-24 20:19:06 - market_analysis - INFO - 
Number of sudden price changes in Spot data: 6782
2026-01-24 20:19:06 - market_analysis - INFO - Number of sudden price changes in Perpetual data: 12108


#### 3.2 Proposed Machine Learning Models for Ultra-Short-Term Price Prediction


In [19]:
# The goal is to predict if the target market will move by at least 0.07% in the next 5ms [User Query].
# This is framed as a multi-class classification problem (Up, Down, No Change).

def prepare_data_for_ml(source_df, target_df, prediction_horizon_ms=5, change_threshold_bps=7):
    """
    Prepares features and labels for ML model.
    Features: recent price changes, spread, OFI from source market.
    Labels: future price movement in target market.
    """
    # Align dataframes by resampling to a common, fine-grained frequency (e.g., 1ms) and ffill.
    # This is a practical compromise for ML models that prefer regular time series.
    # For production, consider pd.merge_asof or custom time-alignment for tick data.[12, 13]
    
    min_time = min(source_df.index.min(), target_df.index.min())
    max_time = max(source_df.index.max(), target_df.index.max())
    
    common_index = pd.date_range(start=min_time, end=max_time, freq='1ms')
    
    # source_resampled = source_df.reindex(common_index, method='ffill')
    # Reindexing with a method requires a unique index.
    # Handle potential duplicate timestamps by keeping the last entry for each.
    if not source_df.index.is_unique:
        source_df = source_df.loc[~source_df.index.duplicated(keep='last')]
    if not target_df.index.is_unique:
        target_df = target_df.loc[~target_df.index.duplicated(keep='last')]

    source_resampled = source_df.reindex(common_index, method='ffill')
    target_resampled = target_df.reindex(common_index, method='ffill')
    target_resampled = target_df.reindex(common_index, method='ffill')
    
    source_resampled.dropna(inplace=True)
    target_resampled.dropna(inplace=True)

    common_time_intersection = source_resampled.index.intersection(target_resampled.index)
    source_resampled = source_resampled.loc[common_time_intersection]
    target_resampled = target_resampled.loc[common_time_intersection]

    # Features from source market
    features = pd.DataFrame(index=source_resampled.index)
    features['source_mid_price_change'] = source_resampled['mid_price'].diff().fillna(0)
    features['source_bid_ask_spread'] = source_resampled['bid_ask_spread'].fillna(0)
    features['source_best_bid_change'] = source_resampled['best_bid_change'].fillna(0)
    features['source_best_ask_change'] = source_resampled['best_ask_change'].fillna(0)
    
    # Add OFI if available in the source_df (only for spot in this example)
    if 'OFI' in source_df.columns:
        # Reindex OFI to the common_index and ffill
        ofi_resampled = source_df['OFI'].reindex(common_index, method='ffill').fillna(0)
        features['source_OFI'] = ofi_resampled.loc[common_time_intersection]
    else:
        features['source_OFI'] = 0 # Placeholder if OFI not applicable/calculated for this source

    # Labels: Future price movement in target market within prediction_horizon_ms
    # 0: no significant change, 1: up by threshold, -1: down by threshold
    
    # Shift target mid-price to get the price at the prediction horizon
    # Use reindex with 'nearest' and tolerance to find the closest future price [19]
    future_mid_price_series = target_resampled['mid_price'].shift(-prediction_horizon_ms) # Simple shift for resampled data
    
    # Calculate the percentage change from current mid_price to future mid_price
    target_future_pct_change = (future_mid_price_series - target_resampled['mid_price']) / target_resampled['mid_price'] * 10000 # in bps

    labels = pd.Series(0, index=target_resampled.index)
    labels[target_future_pct_change >= change_threshold_bps] = 1 # Upward move
    labels[target_future_pct_change <= -change_threshold_bps] = -1 # Downward move
    
    valid_feature_indices = features.dropna().index
    valid_label_indices = labels.dropna().index
    
    common_valid_indices = valid_feature_indices.intersection(valid_label_indices)
    
    # Filter both DataFrames using the common, valid index
    X = features.loc[common_valid_indices]
    y = labels.loc[common_valid_indices]
    
    return X, y




In [20]:
try:
        # Example: Predict Perpetual price movement based on Spot data
    # (Spot is source, Perpetual is target)
    X_spot_to_perpetual, y_spot_to_perpetual = prepare_data_for_ml(spot_data, perpetual_data)

    # Example: Predict Spot price movement based on Perpetual data
    # (Perpetual is source, Spot is target)
    X_perpetual_to_spot, y_perpetual_to_spot = prepare_data_for_ml(perpetual_data, spot_data)

    print(f"\nShape of features (Spot to Perpetual): {X_spot_to_perpetual.shape}")
    print(f"Distribution of labels (Spot to Perpetual):\n{y_spot_to_perpetual.value_counts()}")

except Exception as e:
    logger.error(f"Error preparing data for ML: {str(e)}")
    raise RuntimeError(f"ML preparation failed: {e}") from e


2026-01-24 20:25:24 - market_analysis - ERROR - Error preparing data for ML: Unable to allocate 654. MiB for an array with shape (85731032, 1) and data type float64


RuntimeError: ML preparation failed: Unable to allocate 654. MiB for an array with shape (85731032, 1) and data type float64

#### 3.3 Development and Evaluation of Trading Signals


In [ ]:
# Train and evaluate a classification model (e.g., SVM).[6, 7, 8]
# For real HFT, consider CNNs or LSTMs for their ability to learn complex patterns in time series.[20, 7]

def train_and_evaluate_model(X, y, model_name="SVM"):
    """Trains and evaluates a classification model using accuracy, precision, recall, F1-score."""
    if X.empty or y.empty or len(y.unique()) < 2:
        print(f"Not enough data or classes to train for {model_name}.")
        return None, None
        
    # Stratify to ensure representation of all classes in train/test splits [9]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    
    if model_name == "SVM":
        model = SVC(kernel='rbf', random_state=42, probability=True) # RBF kernel for non-linear data [6, 8]
    # You can add other models here:
    # elif model_name == "RandomForest":
    #     from sklearn.ensemble import RandomForestClassifier
    #     model = RandomForestClassifier(random_state=42)
    # elif model_name == "CNN" or model_name == "LSTM":
    #     # These require deep learning frameworks like TensorFlow/Keras or PyTorch
    #     # and more complex data reshaping (e.g., 3D input for sequences).[20, 7]
    #     print(f"Deep Learning models ({model_name}) require specific setup not fully covered in this example.")
    #     return None, None
    else:
        raise ValueError("Unsupported model type.")
        
    print(f"\nTraining {model_name} model...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    # Precision, Recall, F1-score for specific classes (Up: 1, Down: -1) [9]
    precision_up = precision_score(y_test, y_pred, labels=[1], average='micro', zero_division=0)
    recall_up = recall_score(y_test, y_pred, labels=[1], average='micro', zero_division=0)
    f1_up = f1_score(y_test, y_pred, labels=[1], average='micro', zero_division=0)
    
    precision_down = precision_score(y_test, y_pred, labels=[-1], average='micro', zero_division=0)
    recall_down = recall_score(y_test, y_pred, labels=[-1], average='micro', zero_division=0)
    f1_down = f1_score(y_test, y_pred, labels=[-1], average='micro', zero_division=0)
    
    print(f"--- {model_name} Performance ---")
    print(f"Overall Accuracy: {accuracy:.4f}")
    print(f"Precision (Up): {precision_up:.4f}, Recall (Up): {recall_up:.4f}, F1-Score (Up): {f1_up:.4f}")
    print(f"Precision (Down): {precision_down:.4f}, Recall (Down): {recall_down:.4f}, F1-Score (Down): {f1_down:.4f}")
    
    return model, y_pred

# Train and evaluate models for both directions
print("\n--- Training Spot to Perpetual Prediction Model ---")
svm_spot_to_perpetual, y_pred_spot_to_perpetual = train_and_evaluate_model(X_spot_to_perpetual, y_spot_to_perpetual)

print("\n--- Training Perpetual to Spot Prediction Model ---")
svm_perpetual_to_spot, y_pred_perpetual_to_spot = train_and_evaluate_model(X_perpetual_to_spot, y_perpetual_to_spot)


# --- 4. Characterizing Noise and Momentum in High-Frequency Data ---


#### 4.1 Identification and Statistical Characterization of Noise Events


In [ ]:
# Noise: sudden price changes that revert quickly without affecting the other market [User Query].

def identify_noise_events(df, price_col, sudden_change_mask, reversion_window_ms=10, reversion_threshold_bps=5):
    """
    Identifies noise events: a sudden change that reverts within a window.
    Simplified: checks if price returns close to its pre-change level.
    """
    noise_events = pd.Series(False, index=df.index)
    
    # Iterate through detected sudden changes
    for idx in df[sudden_change_mask].index:
        # Get price before the sudden change
        pre_change_price_idx = df.index.get_loc(idx) - 1
        if pre_change_price_idx < 0:
            continue
        pre_change_price = df[price_col].iloc[pre_change_price_idx]
        
        # Define the time window for reversion check
        reversion_end_time = idx + timedelta(milliseconds=reversion_window_ms)
        
        # Find all prices within the reversion window (inclusive of start, exclusive of end)
        # Use.loc for label-based slicing, which is robust with datetime indices
        window_prices = df.loc[idx:reversion_end_time][price_col]
        
        if not window_prices.empty:
            # Check if any price in the window is close to the pre-change price
            # within the reversion_threshold_bps
            reverted = (np.abs(window_prices - pre_change_price) / pre_change_price * 10000) <= reversion_threshold_bps
            if reverted.any():
                noise_events.loc[idx] = True
                
    return noise_events

# Identify noise events in spot data (as an example)
spot_data['noise_event'] = identify_noise_events(spot_data, 'mid_price', spot_data['sudden_change'])
print(f"\nNumber of noise events detected in Spot data: {spot_data['noise_event'].sum()}")

# For full characterization, you'd analyze:
# - Average frequency of noise events.
# - Average reversion time and price delta of reversion.
# - Attempt to classify types of noise (bid-ask bounce, temporary imbalance, latency artifacts).[21]

# 4.2 Techniques for Filtering Market Microstructure Noise
# Median filter is effective for impulse noise.[10, 11]
# It's best applied to regularly sampled data, so we resample first.

# Resample to a regular frequency (e.g., 1ms) for median filter application.
# This creates a dense series, filling gaps with previous values.
spot_mid_price_resampled = spot_data['mid_price'].resample('1ms').ffill()
perpetual_mid_price_resampled = perpetual_data['mid_price'].resample('1ms').ffill()

# Apply a median filter (kernel_size must be odd).[10]
# A window of 3 or 5 is common for high-frequency data.
window_size = 5 # 5ms window for median filter
spot_data['mid_price_filtered'] = medfilt(spot_mid_price_resampled.values, kernel_size=window_size)
perpetual_data['mid_price_filtered'] = medfilt(perpetual_mid_price_resampled.values, kernel_size=window_size)

print(f"\nOriginal Spot Mid-Price (first 10, resampled): {spot_mid_price_resampled.iloc[:10].tolist()}")
print(f"Filtered Spot Mid-Price (first 10, resampled): {spot_data['mid_price_filtered'][:10].tolist()}")


#### 4.3 Analysis of Momentum Quality and Persistence


In [ ]:
# Quantify how long a price movement sustains its momentum (time duration or price delta) [User Query].

def analyze_momentum_persistence(df, price_col, sudden_change_mask, momentum_window_ms=10, initial_change_threshold_bps=7):
    """
    Analyzes how long a sudden price change sustains its momentum.
    Returns average duration and average additional price delta.
    """
    durations =
    price_deltas =

    # Iterate through detected sudden changes
    for idx in df[sudden_change_mask].index:
        # Get the price at the moment of sudden change
        initial_price = df.loc[idx, price_col]
        
        # Determine the initial direction of the sudden change
        prev_idx_loc = df.index.get_loc(idx) - 1
        if prev_idx_loc < 0:
            continue
        price_before_change = df[price_col].iloc[prev_idx_loc]
        
        price_diff_bps = (initial_price - price_before_change) / price_before_change * 10000
        
        initial_direction = 0
        if price_diff_bps >= initial_change_threshold_bps:
            initial_direction = 1 # Upward
        elif price_diff_bps <= -initial_change_threshold_bps:
            initial_direction = -1 # Downward
        else:
            continue # Not a clear directional sudden change based on threshold
            
        # Look for persistence within the momentum window
        momentum_end_time = idx + timedelta(milliseconds=momentum_window_ms)
        
        # Slice the data from the point of sudden change up to the momentum_end_time
        momentum_segment = df.loc[idx:momentum_end_time]
        
        if momentum_segment.empty:
            continue

        # Find the maximum/minimum price reached within the momentum window
        if initial_direction == 1: # Upward momentum
            peak_price = momentum_segment[price_col].max()
            peak_time = momentum_segment[price_col].idxmax()
            
            # Additional delta beyond the initial price
            additional_delta = (peak_price - initial_price) / initial_price * 10000
            
            if additional_delta > 0: # Only count if it continued in the same direction
                durations.append((peak_time - idx).total_seconds() * 1000)
                price_deltas.append(additional_delta)
                
        elif initial_direction == -1: # Downward momentum
            trough_price = momentum_segment[price_col].min()
            trough_time = momentum_segment[price_col].idxmin()
            
            # Additional delta (positive if it went further down)
            additional_delta = (initial_price - trough_price) / initial_price * 10000
            
            if additional_delta > 0: # Only count if it continued in the same direction
                durations.append((trough_time - idx).total_seconds() * 1000)
                price_deltas.append(additional_delta)
                
    return np.mean(durations) if durations else 0, np.mean(price_deltas) if price_deltas else 0

avg_momentum_duration_ms, avg_price_delta_persistence_bps = analyze_momentum_persistence(
    spot_data, 'mid_price', spot_data['sudden_change'], momentum_window_ms=10
)

print(f"\nAverage Momentum Duration (Spot): {avg_momentum_duration_ms:.2f} ms")
print(f"Average Price Delta Persistence (Spot): {avg_price_delta_persistence_bps:.2f} bps")

# Correlation: Initial Trade Volume vs. Momentum Persistence [User Query]
# This requires aligning the `signed_trade_volume` (or `trade_quantity`) at the time of the
# sudden change with the calculated `durations` and `price_deltas`.
# This is a more advanced statistical analysis. Conceptually:
# 1. For each `sudden_change` event, record the `trade_quantity` at that exact timestamp.
# 2. Pair this quantity with the `duration` and `price_delta` calculated for that specific event.
# 3. Compute the correlation coefficient between these paired lists.

# Example (conceptual, requires careful alignment of individual event data):
# initial_trade_volumes_at_change =
# for idx in spot_data[spot_data['sudden_change']].index:
#     if idx in spot_data.index: # Ensure the index exists after any filtering/resampling
#         initial_trade_volumes_at_change.append(spot_data.loc[idx, 'trade_quantity'])
# # Then calculate correlation between initial_trade_volumes_at_change and the 'durations'/'price_deltas' lists.
# # This requires the lists to be of the same length and correspond to the same events.

print("\n--- End of Code Demonstration ---")
print("This code provides a foundational implementation for the concepts discussed in the research report.")
print("Remember that real-world HFT requires highly optimized data pipelines, advanced statistical methods,")
print("and rigorous backtesting with real market data. Consider these next steps:")
print("1. **Data Infrastructure:** Use a time-series database like QuestDB [22] for efficient data handling.")
print("2. **Advanced ML Models:** Explore CNNs or LSTMs for better predictive power.[20, 7]")
print("3. **Real-time Processing:** Optimize for ultra-low latency, potentially using co-location.[1]")
print("4. **Robust OFI:** If available, use Level 2 order book data for more accurate Order Flow Imbalance.[17]")
print("5. **Adaptive Strategies:** Implement dynamic thresholds for sudden price changes and shifting market leadership.[18]")
print("6. **Rigorous Backtesting:** Test your strategies thoroughly on historical data to validate profitability and risk.")

Fuentes y contenido relacionado


# DeepSearch

## simulate data


In [ ]:
# --- 0. Helper Functions and Simulated Data Generation ---
# In a real scenario, you would load your 24 hours of TRB/USDT spot and perpetual data.
# For demonstration, we'll simulate some high-frequency-like data.

def generate_simulated_tick_data(num_entries, start_time, market_type="spot"):
    """Generates simulated tick data with irregular timestamps."""
    data = 
    current_time = start_time
    base_price = 100.0 if market_type == "spot" else 100.1 # Slight difference for perpetual
    
    for i in range(num_entries):
        # Irregular timestamps: sometimes multiple in 1ms, sometimes gaps
        time_delta_ms = np.random.choice(, p=[0.2, 0.3, 0.2, 0.1, 0.05, 0.05, 0.05, 0.05])
        current_time += timedelta(microseconds=time_delta_ms * 1000)
        
        # Simulate price movement
        price_change = np.random.uniform(-0.05, 0.05) # Small random price changes
        current_price = base_price * (1 + price_change / 1000)
        
        # Simulate bid/ask spread
        spread = np.random.uniform(0.01, 0.05)
        best_bid = current_price - spread / 2
        best_ask = current_price + spread / 2
        
        # Simulate trade data (for spot market only in this example)
        trade_price = np.random.uniform(best_bid, best_ask)
        trade_quantity = np.random.randint(1, 100)
        
        entry = {
            'timestamp': current_time,
            'best_bid': best_bid,
            'best_ask': best_ask,
            'mid_price': (best_bid + best_ask) / 2,
            'trade_price': trade_price if market_type == "spot" else np.nan,
            'trade_quantity': trade_quantity if market_type == "spot" else np.nan
        }
        data.append(entry)
        base_price = current_price # Price evolves
        
    df = pd.DataFrame(data)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp').sort_index()
    return df

# Generate 24 hours worth of data (approx. 86.4 million ms, so a subset for simulation)
# Let's simulate 100,000 entries for each market for demonstration
start_time = datetime(2023, 1, 1, 9, 0, 0)
spot_data = generate_simulated_tick_data(100000, start_time, market_type="spot")
perpetual_data = generate_simulated_tick_data(100000, start_time, market_type="perpetual")

print("Simulated Spot Data Head:")
print(spot_data.head())
print("\nSimulated Perpetual Data Head:")
print(perpetual_data.head())


SyntaxError: invalid syntax (199688215.py, line 7)

## gemini

In [ ]:


# --- 1. Data Pre-processing and Feature Engineering ---

print("\n--- 1. Data Pre-processing and Feature Engineering ---")

# 1.1 Addressing Irregular Timestamps and Data Synchronization Challenges
# The Hayashi-Yoshida estimator (used in section 3.1) inherently handles asynchronous data.
# For other analyses, we might need to resample or use methods robust to irregular timestamps.
# For simplicity in feature engineering, we'll use `ffill` to propagate last known values
# for features that might be calculated across different timestamps.
# In a real HFT system, you'd use a time-series database like QuestDB [6] for efficient handling.

# 1.2 Definition and Calculation of Mid-Price, Bid/Ask Spreads, and Other Microstructure Features
# Mid-price is already calculated in the simulation.
spot_data['bid_ask_spread'] = spot_data['best_ask'] - spot_data['best_bid']
perpetual_data['bid_ask_spread'] = perpetual_data['best_ask'] - perpetual_data['best_bid']

# Example of a "time-sensitive" feature: Price change over a short window (e.g., 1ms)
# This requires careful handling of irregular timestamps. For simplicity, we'll use `diff`
# and then consider the time difference.
spot_data['mid_price_change_1ms'] = spot_data['mid_price'].diff()
perpetual_data['mid_price_change_1ms'] = perpetual_data['mid_price'].diff()

# The user query mentions: "when prices are moving upward, the ask side tends to update first;
# conversely, when prices are declining, the bid side often reacts earlier."
# This can be incorporated by creating features that capture these asymmetric movements.
# For example, tracking changes in best_ask vs best_bid separately.
spot_data['best_bid_change'] = spot_data['best_bid'].diff()
spot_data['best_ask_change'] = spot_data['best_ask'].diff()
perpetual_data['best_bid_change'] = perpetual_data['best_bid'].diff()
perpetual_data['best_ask_change'] = perpetual_data['best_ask'].diff()

print("\nFeatures added to Spot Data (head):")
print(spot_data[['mid_price', 'bid_ask_spread', 'mid_price_change_1ms', 'best_bid_change', 'best_ask_change']].head())

# 1.3 Leveraging Trade Data: Calculation and Predictive Power of Order Flow Imbalance (OFI)
# Based on [9], we can calculate Volume Order Imbalance (VOI).
# This requires previous bid/ask prices and volumes.
# Assuming 'best_bid', 'best_ask' are prices and we need 'best_bid_volume', 'best_ask_volume'
# which are not in the simulated data, so we'll simulate them or use a simplified approach.
# For a more robust OFI, you'd need Level 2 data (order book depth).[7, 8]

def calculate_voi(df):
    """
    Calculates Volume Order Imbalance (VOI) based on.[9]
    Requires 'best_bid', 'best_ask', 'best_bid_volume', 'best_ask_volume'.
    For simulation, we'll assume 'best_bid_volume' and 'best_ask_volume' are available.
    """
    df_copy = df.copy()
    
    # Simulate bid/ask volumes for demonstration
    df_copy['best_bid_volume'] = np.random.randint(100, 1000, size=len(df_copy))
    df_copy['best_ask_volume'] = np.random.randint(100, 1000, size=len(df_copy))

    df_copy['prev_best_bid'] = df_copy['best_bid'].shift(1)
    df_copy['prev_best_ask'] = df_copy['best_ask'].shift(1)
    df_copy['prev_best_bid_volume'] = df_copy['best_bid_volume'].shift(1)
    df_copy['prev_best_ask_volume'] = df_copy['best_ask_volume'].shift(1)

    delta_vb =
    delta_va =

    for i in range(len(df_copy)):
        if i == 0:
            delta_vb.append(0)
            delta_va.append(0)
            continue

        pb_t = df_copy['best_bid'].iloc[i]
        pb_t_1 = df_copy['prev_best_bid'].iloc[i]
        vb_t = df_copy['best_bid_volume'].iloc[i]
        vb_t_1 = df_copy['prev_best_bid_volume'].iloc[i]

        pa_t = df_copy['best_ask'].iloc[i]
        pa_t_1 = df_copy['prev_best_ask'].iloc[i]
        va_t = df_copy['best_ask_volume'].iloc[i]
        va_t_1 = df_copy['prev_best_ask_volume'].iloc[i]

        # Calculate delta_VB
        if pd.isna(pb_t) or pd.isna(pb_t_1) or pd.isna(vb_t) or pd.isna(vb_t_1):
            dvb = 0
        elif pb_t < pb_t_1:
            dvb = 0 # Conservative approach [9]
        elif pb_t == pb_t_1:
            dvb = vb_t - vb_t_1
        elif pb_t > pb_t_1:
            dvb = vb_t # Upward price momentum [9]
        else:
            dvb = 0 # Should not happen with proper data

        # Calculate delta_VA
        if pd.isna(pa_t) or pd.isna(pa_t_1) or pd.isna(va_t) or pd.isna(va_t_1):
            dva = 0
        elif pa_t < pa_t_1:
            dva = va_t # Downward price momentum [9]
        elif pa_t == pa_t_1:
            dva = va_t - va_t_1
        elif pa_t > pa_t_1:
            dva = 0 # Conservative approach [9]
        else:
            dva = 0 # Should not happen with proper data
            
        delta_vb.append(dvb)
        delta_va.append(dva)

    df_copy = delta_vb
    df_copy['delta_VA'] = delta_va
    df_copy['VOI'] = df_copy - df_copy['delta_VA'] # [9]
    
    return df_copy

# Apply VOI calculation to spot data (as it has trade data)
spot_data_with_voi = calculate_voi(spot_data)
print("\nSpot Data with VOI (head):")
print(spot_data_with_voi[['mid_price', 'VOI']].head())


# --- 2. Quantifying Lead-Lag Relationships and Price Discovery ---

print("\n--- 2. Quantifying Lead-Lag Relationships and Price Discovery ---")

# 2.1 Methodology for Detecting Lead-Lag Relationships Between Spot and Perpetual Markets
# Using the 'lead_lag' library for Hayashi-Yoshida estimator [10, 11, 2]
# It handles non-synchronous data.[2]

# For demonstration, let's take a subset of data to make computation faster
# In a real scenario, you'd run this over the full 24 hours.
sample_size = 10000
spot_mid_prices_sample = spot_data['mid_price'].iloc[:sample_size]
perpetual_mid_prices_sample = perpetual_data['mid_price'].iloc[:sample_size]

# Ensure indices are aligned for the lead_lag function, even if timestamps are irregular
# The lead_lag library expects pandas Series with datetime index [2]
spot_series = pd.Series(spot_mid_prices_sample.values, index=spot_mid_prices_sample.index)
perpetual_series = pd.Series(perpetual_mid_prices_sample.values, index=perpetual_mid_prices_sample.index)

# Calculate lead-lag for mid-prices
# max_lag should be set based on the expected 3-5ms window. Let's use 10ms (0.01 seconds) [2]
estimated_lag_spot_perpetual = lag(spot_series, perpetual_series, max_lag=0.01) # [2]

print(f"\nEstimated Lead-Lag (Spot vs Perpetual Mid-Price): {estimated_lag_spot_perpetual * 1000:.2f} ms")
if estimated_lag_spot_perpetual is not None:
    if estimated_lag_spot_perpetual > 0:
        print("Interpretation: Spot market leads the Perpetual market. [2]")
    elif estimated_lag_spot_perpetual < 0:
        print("Interpretation: Perpetual market leads the Spot market. [2]")
    else:
        print("Interpretation: No significant lead-lag detected within the max_lag window.")
else:
    print("Could not estimate lead-lag.")

# 2.2 Analysis of Which Market Drives Price Discovery More Often
# This requires analyzing the lead-lag over different periods and correlating with liquidity metrics.
# For a full analysis, you'd iterate this over time windows and record results in a table as suggested in the report.
# Example: Calculate average bid-ask spread and trade volume for the sample
avg_spot_spread = spot_data['bid_ask_spread'].mean()
avg_perpetual_spread = perpetual_data['bid_ask_spread'].mean()
avg_spot_trade_volume = spot_data['trade_quantity'].mean()

print(f"\nAverage Spot Bid-Ask Spread: {avg_spot_spread:.4f}")
print(f"Average Perpetual Bid-Ask Spread: {avg_perpetual_spread:.4f}")
print(f"Average Spot Trade Volume: {avg_spot_trade_volume:.2f}")

# In a real scenario, you'd perform this analysis over many time windows
# and aggregate results to fill the table from the research report.
# The market with higher liquidity (e.g., tighter spreads, higher volume) tends to lead.[10]


# --- 3. Predictive Modeling for Cross-Market Trading Signals ---

print("\n--- 3. Predictive Modeling for Cross-Market Trading Signals ---")

# 3.1 Defining and Detecting "Sudden Price Changes"
# User-defined: ±0.07% (7 bps) price move within the last 3 ms [User Query]

def detect_sudden_price_change(df, price_col, threshold_bps=7, time_window_ms=3):
    """
    Detects sudden price changes based on a percentage move within a time window.
    Returns a boolean Series indicating where a sudden change occurred.
    """
    df_copy = df.copy()
    df_copy['price_prev'] = df_copy[price_col].shift(1)
    df_copy['time_prev'] = df_copy.index.to_series().shift(1)
    
    # Calculate percentage change and time difference for each tick
    df_copy['pct_change'] = (df_copy[price_col] - df_copy['price_prev']) / df_copy['price_prev'] * 10000 # in bps
    df_copy['time_diff_ms'] = (df_copy.index.to_series() - df_copy['time_prev']).dt.total_seconds() * 1000
    
    # Identify points where the change happened within the specified time window
    # and the magnitude exceeds the threshold
    sudden_change_mask = (df_copy['time_diff_ms'] <= time_window_ms) & \
                         (df_copy['pct_change'].abs() >= threshold_bps)
    
    return sudden_change_mask.fillna(False)

# Detect sudden price changes in spot and perpetual mid-prices
spot_data['sudden_change'] = detect_sudden_price_change(spot_data, 'mid_price')
perpetual_data['sudden_change'] = detect_sudden_price_change(perpetual_data, 'mid_price')

print(f"\nNumber of sudden price changes in Spot data: {spot_data['sudden_change'].sum()}")
print(f"Number of sudden price changes in Perpetual data: {perpetual_data['sudden_change'].sum()}")

# 3.2 Proposed Machine Learning Models for Ultra-Short-Term Price Prediction
# We'll use a simplified approach for demonstration: predict if the target market
# will move by at least 0.07% in the next 5ms. This is a classification problem [User Query].

def prepare_data_for_ml(source_df, target_df, prediction_horizon_ms=5, change_threshold_bps=7):
    """
    Prepares features and labels for ML model.
    Features: recent price changes, spread, OFI from source market.
    Labels: future price movement in target market.
    """
    # Align dataframes by time. This is a simplification; in real HFT, you'd use a merge_asof or similar.
    # For this example, we'll resample to a common, fine-grained frequency (e.g., 1ms) and ffill.
    # This is a compromise for demonstration, as true tick data is irregular.
    
    # Create a common time index for merging
    min_time = min(source_df.index.min(), target_df.index.min())
    max_time = max(source_df.index.max(), target_df.index.max())
    
    # Create a regular 1ms index
    common_index = pd.date_range(start=min_time, end=max_time, freq='1ms')
    
    source_resampled = source_df.reindex(common_index, method='ffill')
    target_resampled = target_df.reindex(common_index, method='ffill')
    
    # Drop NaNs introduced by reindexing before the first actual data point
    source_resampled.dropna(inplace=True)
    target_resampled.dropna(inplace=True)

    # Ensure both dataframes cover the same time range after resampling and dropping NaNs
    # Find the intersection of their indices
    common_time_intersection = source_resampled.index.intersection(target_resampled.index)
    source_resampled = source_resampled.loc[common_time_intersection]
    target_resampled = target_resampled.loc[common_time_intersection]

    # Features from source market (e.g., mid-price change, spread, OFI)
    # We need to ensure OFI is calculated on the source_df before resampling if it depends on previous values.
    # For this example, we'll use simple features from the resampled data.
    features = pd.DataFrame(index=source_resampled.index)
    features['source_mid_price_change'] = source_resampled['mid_price'].diff().fillna(0)
    features['source_bid_ask_spread'] = source_resampled['bid_ask_spread'].fillna(0)
    
    # Add OFI if available (from spot_data_with_voi)
    if 'VOI' in source_df.columns:
        # Reindex VOI to the common_index and ffill
        voi_resampled = source_df['VOI'].reindex(common_index, method='ffill').fillna(0)
        features['source_VOI'] = voi_resampled.loc[common_time_intersection]
    else:
        features['source_VOI'] = 0 # Placeholder if not calculated for this source

    # Labels: Future price movement in target market
    # Predict if mid_price moves up/down by threshold_bps within prediction_horizon_ms
    
    # Calculate future mid-price for the target market
    future_mid_price_idx = target_resampled.index + timedelta(milliseconds=prediction_horizon_ms)
    
    # Use reindex with 'nearest' to find the closest future price within the horizon
    # This is a simplified way to get the price *at or near* the prediction horizon.
    # A more robust approach would involve looking up the first price *after* the horizon.
    
    # Create a temporary series for future prices aligned with current timestamps
    future_prices_aligned = target_resampled['mid_price'].reindex(future_mid_price_idx, method='nearest', tolerance=timedelta(milliseconds=prediction_horizon_ms))
    
    # Calculate the percentage change from current mid_price to future mid_price
    target_future_pct_change = (future_prices_aligned - target_resampled['mid_price']) / target_resampled['mid_price'] * 10000 # in bps

    # Define labels: 0 for no significant change, 1 for up, -1 for down
    labels = pd.Series(0, index=target_resampled.index)
    labels[target_future_pct_change >= change_threshold_bps] = 1 # Upward move
    labels[target_future_pct_change <= -change_threshold_bps] = -1 # Downward move
    
    # Drop any rows where future price could not be found (e.g., at the very end of the data)
    # or where features are NaN (e.g., first row after diff)
    combined_df = pd.concat([features, labels.rename('label')], axis=1).dropna()
    
    X = combined_df.drop('label', axis=1)
    y = combined_df['label']
    
    return X, y

# Example: Predict Perpetual price movement based on Spot data
X_spot_to_perpetual, y_spot_to_perpetual = prepare_data_for_ml(spot_data_with_voi, perpetual_data)

# Example: Predict Spot price movement based on Perpetual data
X_perpetual_to_spot, y_perpetual_to_spot = prepare_data_for_ml(perpetual_data, spot_data_with_voi)

print(f"\nShape of features (Spot to Perpetual): {X_spot_to_perpetual.shape}")
print(f"Distribution of labels (Spot to Perpetual):\n{y_spot_to_perpetual.value_counts()}")

# Train a simple SVM model [3, 12, 4, 13]
# For real HFT, you'd use more advanced models like CNNs or LSTMs
# and more sophisticated feature engineering.

def train_and_evaluate_model(X, y, model_name="SVM"):
    """Trains and evaluates a classification model."""
    if X.empty or y.empty:
        print(f"No data to train for {model_name}.")
        return None, None
        
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    
    if model_name == "SVM":
        model = SVC(kernel='rbf', random_state=42) # RBF kernel for non-linear data [3]
    # You could add other models here:
    # elif model_name == "RandomForest":
    #     from sklearn.ensemble import RandomForestClassifier
    #     model = RandomForestClassifier(random_state=42)
    # elif model_name == "LogisticRegression":
    #     from sklearn.linear_model import LogisticRegression
    #     model = LogisticRegression(max_iter=1000, random_state=42)
    else:
        raise ValueError("Unsupported model type.")
        
    print(f"\nTraining {model_name} model...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision_up = precision_score(y_test, y_pred, labels=[1], average='micro', zero_division=0)
    recall_up = recall_score(y_test, y_pred, labels=[1], average='micro', zero_division=0)
    f1_up = f1_score(y_test, y_pred, labels=[1], average='micro', zero_division=0)
    
    precision_down = precision_score(y_test, y_pred, labels=[-1], average='micro', zero_division=0)
    recall_down = recall_score(y_test, y_pred, labels=[-1], average='micro', zero_division=0)
    f1_down = f1_score(y_test, y_pred, labels=[-1], average='micro', zero_division=0)
    
    print(f"--- {model_name} Performance ---")
    print(f"Overall Accuracy: {accuracy:.4f}")
    print(f"Precision (Up): {precision_up:.4f}, Recall (Up): {recall_up:.4f}, F1-Score (Up): {f1_up:.4f}")
    print(f"Precision (Down): {precision_down:.4f}, Recall (Down): {recall_down:.4f}, F1-Score (Down): {f1_down:.4f}")
    
    return model, y_pred

# Train and evaluate models
print("\n--- Training Spot to Perpetual Prediction Model ---")
svm_spot_to_perpetual, y_pred_spot_to_perpetual = train_and_evaluate_model(X_spot_to_perpetual, y_spot_to_perpetual)

print("\n--- Training Perpetual to Spot Prediction Model ---")
svm_perpetual_to_spot, y_pred_perpetual_to_spot = train_and_evaluate_model(X_perpetual_to_spot, y_perpetual_to_spot)


# --- 4. Characterizing Noise and Momentum in High-Frequency Data ---

print("\n--- 4. Characterizing Noise and Momentum in High-Frequency Data ---")

# 4.1 Identification and Statistical Characterization of Noise Events
# Noise is defined as sudden price changes that revert quickly without affecting the other market [User Query].
# This requires tracking the price after a "sudden change" and checking for reversion.

def identify_noise_events(df, price_col, sudden_change_mask, reversion_window_ms=10, reversion_threshold_bps=5):
    """
    Identifies noise events: a sudden change that reverts within a window.
    Simplified: checks if price returns close to its pre-change level.
    """
    noise_events = pd.Series(False, index=df.index)
    
    # Iterate through detected sudden changes
    for idx in df[sudden_change_mask].index:
        # Get price before the sudden change
        pre_change_price_idx = df.index.get_loc(idx) - 1
        if pre_change_price_idx < 0:
            continue
        pre_change_price = df[price_col].iloc[pre_change_price_idx]
        
        # Look for price within the reversion window
        reversion_end_time = idx + timedelta(milliseconds=reversion_window_ms)
        
        # Find all prices within the reversion window
        window_prices = df.loc[idx:reversion_end_time][price_col]
        
        if not window_prices.empty:
            # Check if any price in the window is close to the pre-change price
            # within the reversion_threshold_bps
            reverted = (np.abs(window_prices - pre_change_price) / pre_change_price * 10000) <= reversion_threshold_bps
            if reverted.any():
                noise_events.loc[idx] = True
                
    return noise_events

# Identify noise events in spot data (as an example)
spot_data['noise_event'] = identify_noise_events(spot_data, 'mid_price', spot_data['sudden_change'])
print(f"\nNumber of noise events detected in Spot data: {spot_data['noise_event'].sum()}")

# 4.2 Techniques for Filtering Market Microstructure Noise
# Median filter is a good choice for impulse noise.[5]
# For demonstration, let's apply it to the mid-price.

# Resample to a regular frequency for median filter application
# This is a common step before applying many signal processing filters.
# Choose a frequency that balances noise reduction and information loss.
# Here, we'll resample to 1ms and then apply the filter.
spot_mid_price_resampled = spot_data['mid_price'].resample('1ms').ffill()
perpetual_mid_price_resampled = perpetual_data['mid_price'].resample('1ms').ffill()

# Apply a median filter (window size must be odd)
# A window of 3 or 5 is common for high-frequency data.
window_size = 5 # 5ms window for median filter
spot_data['mid_price_filtered'] = medfilt(spot_mid_price_resampled.values, kernel_size=window_size)
perpetual_data['mid_price_filtered'] = medfilt(perpetual_mid_price_resampled.values, kernel_size=window_size)

print(f"\nOriginal Spot Mid-Price (first 10): {spot_mid_price_resampled.iloc[:10].tolist()}")
print(f"Filtered Spot Mid-Price (first 10): {spot_data['mid_price_filtered'][:10].tolist()}")

# 4.3 Analysis of Momentum Quality and Persistence
# Quantify how long a price movement sustains its momentum (time duration or price delta) [User Query].

def analyze_momentum_persistence(df, price_col, sudden_change_mask, momentum_window_ms=10, initial_direction_threshold_bps=0.07):
    """
    Analyzes how long a sudden price change sustains its momentum.
    Returns average duration and average additional price delta.
    """
    durations =
    price_deltas =

    # Iterate through detected sudden changes
    for idx in df[sudden_change_mask].index:
        # Get the price at the moment of sudden change
        initial_price = df.loc[idx, price_col]
        
        # Determine the initial direction of the sudden change
        # Need to look at the price just before the change and at the change point
        prev_idx_loc = df.index.get_loc(idx) - 1
        if prev_idx_loc < 0:
            continue
        price_before_change = df[price_col].iloc[prev_idx_loc]
        
        if (initial_price - price_before_change) / price_before_change * 10000 >= initial_direction_threshold_bps:
            initial_direction = 1 # Upward
        elif (initial_price - price_before_change) / price_before_change * 10000 <= -initial_direction_threshold_bps:
            initial_direction = -1 # Downward
        else:
            continue # Not a clear directional sudden change based on threshold
            
        # Look for persistence within the momentum window
        momentum_end_time = idx + timedelta(milliseconds=momentum_window_ms)
        
        # Slice the data from the point of sudden change up to the momentum_end_time
        # Use.loc with inclusive slicing for both start and end
        momentum_segment = df.loc[idx:momentum_end_time]
        
        if momentum_segment.empty:
            continue

        # Find the maximum/minimum price reached within the momentum window
        if initial_direction == 1: # Upward momentum
            peak_price = momentum_segment[price_col].max()
            # Find the time when this peak was reached
            peak_time = momentum_segment[price_col].idxmax()
            
            # Check if price continued to move in the same direction beyond initial change
            additional_delta = (peak_price - initial_price) / initial_price * 10000
            
            # Consider it sustained if it moved further in the same direction
            if additional_delta > 0:
                durations.append((peak_time - idx).total_seconds() * 1000)
                price_deltas.append(additional_delta)
                
        elif initial_direction == -1: # Downward momentum
            trough_price = momentum_segment[price_col].min()
            # Find the time when this trough was reached
            trough_time = momentum_segment[price_col].idxmin()
            
            # Check if price continued to move further down
            additional_delta = (initial_price - trough_price) / initial_price * 10000 # Positive if it went further down
            
            if additional_delta > 0:
                durations.append((trough_time - idx).total_seconds() * 1000)
                price_deltas.append(additional_delta)
                
    return np.mean(durations) if durations else 0, np.mean(price_deltas) if price_deltas else 0

avg_momentum_duration_ms, avg_price_delta_persistence_bps = analyze_momentum_persistence(
    spot_data, 'mid_price', spot_data['sudden_change'], momentum_window_ms=10
)

print(f"\nAverage Momentum Duration (Spot): {avg_momentum_duration_ms:.2f} ms")
print(f"Average Price Delta Persistence (Spot): {avg_price_delta_persistence_bps:.2f} bps")

# Correlation with initial trade volume (conceptual, as simulated data is simple)
# In a real scenario, you'd align trade data with sudden changes and calculate correlation.
# For example, if you have a 'trade_quantity' column aligned with the sudden change,
# you could correlate it with `durations` and `price_deltas`.
# This would involve more complex data alignment and statistical tests.

print("\n--- End of Code Demonstration ---")
print("This code provides a foundational implementation for the concepts discussed in the research report.")
print("Remember that real-world HFT requires highly optimized data pipelines, advanced statistical methods,")
print("and rigorous backtesting with real market data.")